In [1]:
print("Hello World")

Hello World


In [2]:
import numpy as np
import pandas as pd

### Loading Datasets

In [3]:
baseline = pd.read_csv("./Heart Disease Prediction/baseline.csv")

In [4]:
ground_truths = pd.read_csv("./Heart Disease Prediction/Ground Truths/07th Feb 2024.csv")

In [5]:
production_run = pd.read_csv("./Heart Disease Prediction/Production Runs/07th Feb 2024.csv")

In [6]:
baseline.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,target
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
3,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0
4,45,F,ATA,130,237,0,Normal,170,N,0.0,Up,0


In [7]:
ground_truths.head()

,Unnamed: 0,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,target
0,668,63,F,ATA,140,195,0,Normal,179,N,0.0,Up,0
1,30,53,M,NAP,145,518,0,Normal,130,N,0.0,Flat,1
2,377,65,M,ASY,160,0,1,ST,122,N,1.2,Flat,1
3,535,56,M,ASY,130,0,0,LVH,122,Y,1.0,Flat,1
4,807,54,M,ATA,108,309,0,Normal,156,N,0.0,Up,0


In [8]:
production_run.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,target,probs
0,63,F,ATA,140.0,195.0,0.0,Normal,179.0,N,0.0,Up,0,1.0
1,53,M,NAP,145.0,518.0,0.0,Normal,130.0,N,0.0,Flat,1,0.6
2,65,M,ASY,160.0,0.0,1.0,ST,122.0,N,1.2,Flat,1,1.0
3,56,M,ASY,130.0,0.0,0.0,LVH,122.0,Y,1.0,Flat,1,1.0
4,54,M,ATA,108.0,309.0,0.0,Normal,156.0,N,0.0,Up,0,1.0


In [9]:
ground_truths.drop("Unnamed: 0", axis=1, inplace=True)

### Checking the filtering code

In [10]:
baseline[baseline['Oldpeak'].isin(range(2,4))]

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,target
6,58,M,ATA,136,164,0,ST,99,Y,2.0,Flat,1
13,36,M,ATA,120,267,0,Normal,160,N,3.0,Flat,1
17,53,M,ASY,124,260,0,ST,112,Y,3.0,Flat,0
19,54,M,ASY,125,224,0,Normal,122,N,2.0,Flat,1
30,47,F,ASY,120,205,0,Normal,98,Y,2.0,Flat,1
...,...,...,...,...,...,...,...,...,...,...,...,...
403,64,M,ASY,145,212,0,LVH,132,N,2.0,Flat,1
406,53,M,ASY,123,282,0,Normal,95,Y,2.0,Flat,1
467,60,M,NAP,140,185,0,LVH,155,N,3.0,Flat,1
489,57,M,ASY,110,335,0,Normal,143,Y,3.0,Flat,1


In [11]:
from utils import determine_dtype_ft

In [12]:
cat_ft, num_ft = determine_dtype_ft(baseline)

In [13]:
cat_ft, num_ft

(['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope'],
 ['Age',
  'RestingBP',
  'Cholesterol',
  'FastingBS',
  'MaxHR',
  'Oldpeak',
  'target'])

In [14]:
num_conds, cat_conds = dict(), dict()

for i in list(baseline.columns)[:3]:
    cond = input(f"Enter condition for {i}")
    
    if i in cat_ft:
        cat_conds[i] = cond
    if i in num_ft:
        num_conds[i] = cond
    if cond == "No":
        continue


Enter condition for Age30
Enter condition for SexM
Enter condition for ChestPainTypeATA


In [15]:
num_conds, cat_conds

({'Age': '30'}, {'Sex': 'M', 'ChestPainType': 'ATA'})

In [16]:
num_conds['Age'].split('-')

['30']

In [17]:
'30-'.split('-')

['30', '']

In [18]:
'M' in list(baseline['Sex'].unique())

True

In [19]:
not all([list(baseline['Sex'].unique()), True])

False

In [20]:
baseline['Age'].min()

29

In [21]:
num_conds['Age'] = '40'

In [22]:
cat_conds['Sex'] = ['M']
cat_conds['ChestPainType'] = ['ASY', 'ATA']

In [23]:
cat_conds

{'Sex': ['M'], 'ChestPainType': ['ASY', 'ATA']}

In [24]:
def new_filter_data(df, num_conds, cat_conds):
    
    """
    df: DataFrame to filter
    num_conds: Numerical Conditions that we get from webpage
    cat_conds: Categorical Conditions that we get from webpage
    
    Returns the filtered DataFrame
    """
    
    for i in list(cat_conds.keys()):
        unique_cats = list(df[i].unique())
        if not all([j in unique_cats for j in cat_conds[i]]):
            print("Wrong Category Entered, Can't Filter")
            return None
        else:
            df = df[df[i].isin(cat_conds[i])]
    
    for i in list(num_conds.keys()):
        if '-' in num_conds[i]:
            splits = num_conds[i].split('-')
            limits = [df[i].min() if splits[0] == '' else float(splits[0]), df[i].max() if splits[1] == '' else float(splits[1])]
            
            df = df[(df[i]>=limits[0]) & (df[i]<=limits[1])]
        else:
            try:
                df = df[df[i] == float(num_conds[i])]
            except:
                print("Invalid format to filter Numerical Columns")
                return None
    
    return df

In [25]:
filtered_df = new_filter_data(baseline, num_conds, cat_conds)

In [26]:
baseline[baseline['Sex'].isin(cat_conds['Sex'])]

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,target
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
3,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0
6,58,M,ATA,136,164,0,ST,99,Y,2.0,Flat,1
7,49,M,ASY,140,234,0,Normal,140,Y,1.0,Flat,1
10,38,M,ASY,110,196,0,Normal,166,N,0.0,Flat,1
...,...,...,...,...,...,...,...,...,...,...,...,...
494,56,M,ATA,120,240,0,Normal,169,N,0.0,Down,0
495,67,M,NAP,152,212,0,LVH,150,N,0.8,Flat,1
497,63,M,ASY,140,187,0,LVH,144,Y,4.0,Up,1
498,68,M,ASY,144,193,1,Normal,141,N,3.4,Flat,1


In [27]:
unique_cats = list(baseline['Sex'].unique())

In [28]:
[j in unique_cats for j in cat_conds['Sex']]

[True]

In [29]:
all([j in unique_cats for j in cat_conds['Sex']])

True

In [30]:
cat_conds['Sex']

['M']

In [31]:
filtered_df

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,target
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
50,40,M,ASY,120,466,1,Normal,152,Y,1.0,Flat,1
233,40,M,ASY,125,0,1,Normal,165,N,0.0,Flat,1


In [32]:
filtered_df['Age'].max(), filtered_df['Age'].min()

(40, 40)

In [33]:
filtered_df['Sex'].unique(), filtered_df['ChestPainType'].unique()

(array(['M'], dtype=object), array(['ATA', 'ASY'], dtype=object))

In [34]:
baseline.max()

Age                77
Sex                 M
ChestPainType      TA
RestingBP         200
Cholesterol       603
FastingBS           1
RestingECG         ST
MaxHR             192
ExerciseAngina      Y
Oldpeak           6.2
ST_Slope           Up
target              1
dtype: object

In [35]:
from utils import classification_metrics

In [36]:
filtered_gt = new_filter_data(ground_truths, num_conds, cat_conds)

In [37]:
filtered_pr = new_filter_data(production_run, num_conds, cat_conds)

In [38]:
sample = filtered_pr['target']

In [39]:
sample[31] = 1

In [40]:
filtered_gt['target'], filtered_pr['target']

(31     1
 98     1
 176    1
 Name: target, dtype: int64,
 31     1
 98     1
 176    1
 Name: target, dtype: int64)

In [41]:
results = classification_metrics(filtered_gt['target'], filtered_pr['target'])

ValueError: Only one class present in y_true. ROC AUC score is not defined in that case.

In [42]:
results

NameError: name 'results' is not defined

In [56]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [44]:
accuracy_score([0, 0, 0], [0, 0, 1])

0.6666666666666666

In [45]:
precision_score(filtered_gt['target'], sample)

1.0

In [46]:
recall_score(filtered_gt['target'], sample)

1.0

In [47]:
f1_score(filtered_gt['target'], sample)

1.0

In [48]:
production_run.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,target,probs
0,63,F,ATA,140.0,195.0,0.0,Normal,179.0,N,0.0,Up,0,1.0
1,53,M,NAP,145.0,518.0,0.0,Normal,130.0,N,0.0,Flat,1,0.6
2,65,M,ASY,160.0,0.0,1.0,ST,122.0,N,1.2,Flat,1,1.0
3,56,M,ASY,130.0,0.0,0.0,LVH,122.0,Y,1.0,Flat,1,1.0
4,54,M,ATA,108.0,309.0,0.0,Normal,156.0,N,0.0,Up,0,1.0


In [49]:
ground_truths.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,target
0,63,F,ATA,140,195,0,Normal,179,N,0.0,Up,0
1,53,M,NAP,145,518,0,Normal,130,N,0.0,Flat,1
2,65,M,ASY,160,0,1,ST,122,N,1.2,Flat,1
3,56,M,ASY,130,0,0,LVH,122,Y,1.0,Flat,1
4,54,M,ATA,108,309,0,Normal,156,N,0.0,Up,0


In [50]:
filtered_gt = new_filter_data(ground_truths, num_conds, cat_conds)

In [51]:
filtered_pr = new_filter_data(production_run, num_conds, cat_conds)

In [52]:
filtered_gt.shape, filtered_pr.shape

((3, 12), (3, 13))

In [53]:
filtered_gt

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,target
31,40,M,ASY,152,223,0,Normal,181,N,0.0,Up,1
98,40,M,ASY,110,167,0,LVH,114,Y,2.0,Flat,1
176,40,M,ASY,95,0,1,ST,144,N,0.0,Up,1


In [55]:
filtered_pr

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,target,probs
31,40,M,ASY,152.0,223.0,0.0,Normal,181.0,N,0.0,Up,0,1.0
98,40,M,ASY,110.0,NaN,0.0,LVH,114.0,Y,2.0,Flat,1,1.0
176,40,M,ASY,95.0,0.0,1.0,ST,144.0,N,0.0,Up,1,0.6


In [59]:
print(classification_report(filtered_gt['target'], filtered_pr['target'], output_dict=True))

{'0': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 0}, '1': {'precision': 1.0, 'recall': 0.6666666666666666, 'f1-score': 0.8, 'support': 3}, 'accuracy': 0.6666666666666666, 'macro avg': {'precision': 0.5, 'recall': 0.3333333333333333, 'f1-score': 0.4, 'support': 3}, 'weighted avg': {'precision': 1.0, 'recall': 0.6666666666666666, 'f1-score': 0.8000000000000002, 'support': 3}}


D:\Anaconda\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\Anaconda\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\Anaconda\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [68]:
num_conds['Age'] = '30-50'

In [69]:
cat_conds

{'ChestPainType': ['ASY', 'ATA']}

In [70]:
cat_conds = {'ChestPainType': ['ASY', 'ATA']}

In [71]:
num_conds, cat_conds

({'Age': '30-50'}, {'ChestPainType': ['ASY', 'ATA']})

In [72]:
filtered_gt = new_filter_data(ground_truths, num_conds, cat_conds)
filtered_pr = new_filter_data(production_run, num_conds, cat_conds)

In [73]:
filtered_gt.shape, filtered_pr.shape

((49, 12), (46, 13))

In [77]:
filtered_gt.describe()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,target
count,49.000000,49.000000,49.000000,49.000000,49.000000,49.000000,49.000000
mean,43.244898,127.938776,223.755102,0.122449,144.204082,0.914286,0.571429
std,4.993874,18.697603,103.918544,0.331201,23.996510,0.987632,0.500000
min,32.000000,92.000000,0.000000,0.000000,102.000000,0.000000,0.000000
25%,40.000000,118.000000,193.000000,0.000000,127.000000,0.000000,0.000000
50%,44.000000,122.000000,227.000000,0.000000,144.000000,1.000000,1.000000
75%,48.000000,140.000000,268.000000,0.000000,165.000000,2.000000,1.000000
max,50.000000,190.000000,529.000000,1.000000,188.000000,3.000000,1.000000


In [78]:
filtered_pr.describe()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,target,probs
count,46.000000,46.000000,39.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,43.217391,126.826087,213.564103,0.130435,144.586957,0.865217,0.543478,0.878261
std,5.154989,18.678930,86.989350,0.340503,24.636537,0.946859,0.503610,0.160434
min,32.000000,92.000000,0.000000,0.000000,102.000000,0.000000,0.000000,0.600000
25%,39.250000,115.750000,188.000000,0.000000,125.500000,0.000000,0.000000,0.800000
50%,44.000000,120.000000,225.000000,0.000000,144.000000,0.800000,1.000000,1.000000
75%,48.000000,139.500000,264.500000,0.000000,167.250000,1.575000,1.000000,1.000000
max,50.000000,190.000000,349.000000,1.000000,188.000000,3.000000,1.000000,1.000000


In [79]:
new_gt = ground_truths[ground_truths.index.isin(production_run.index)]

In [80]:
new_gt.shape, production_run.shape

((209, 12), (209, 13))

In [81]:
filtered_gt = new_filter_data(new_gt, num_conds, cat_conds)

In [82]:
filtered_gt.shape, filtered_pr.shape

((49, 12), (46, 13))

In [83]:
ground_truths.shape, production_run.shape

((209, 12), (209, 13))

In [86]:
extra = filtered_gt[~filtered_gt.index.isin(filtered_pr.index)]

In [87]:
extra.shape

(3, 12)

In [94]:
extra

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,target
33,43,M,ASY,150,247,0,Normal,130,Y,2.0,Flat,1
41,44,M,ATA,150,288,0,Normal,150,Y,3.0,Flat,1
46,44,M,ASY,135,491,0,Normal,135,N,0.0,Flat,1


In [93]:
extra.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3 entries, 33 to 46
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             3 non-null      int64  
 1   Sex             3 non-null      object 
 2   ChestPainType   3 non-null      object 
 3   RestingBP       3 non-null      int64  
 4   Cholesterol     3 non-null      int64  
 5   FastingBS       3 non-null      int64  
 6   RestingECG      3 non-null      object 
 7   MaxHR           3 non-null      int64  
 8   ExerciseAngina  3 non-null      object 
 9   Oldpeak         3 non-null      float64
 10  ST_Slope        3 non-null      object 
 11  target          3 non-null      int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 312.0+ bytes


In [90]:
ground_truths.duplicated().sum()

0

In [92]:
production_run.duplicated().sum()

0

In [96]:
ground_truths.iloc[extra.index]

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,target
33,43,M,ASY,150,247,0,Normal,130,Y,2.0,Flat,1
41,44,M,ATA,150,288,0,Normal,150,Y,3.0,Flat,1
46,44,M,ASY,135,491,0,Normal,135,N,0.0,Flat,1


In [97]:
production_run.iloc[extra.index]

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,target,probs
33,43,M,NaN,NaN,NaN,NaN,NaN,NaN,Y,2.0,Flat,1,0.8
41,44,M,NaN,NaN,NaN,NaN,NaN,NaN,Y,3.0,Flat,1,1.0
46,44,M,NaN,NaN,NaN,NaN,NaN,NaN,N,0.0,Flat,1,0.6


In [110]:
classification_report(filtered_gt[filtered_gt.index.isin(filtered_pr.index)]['target'], filtered_pr['target'], output_dict=True)[0]

KeyError: 0

### Checking Profile Report

In [113]:
pip install pydantic-settings

Note: you may need to restart the kernel to use updated packages.


In [125]:
from pydantic_settings import BaseSettings
from ydata_profiling import ProfileReport
from streamlit_pandas_profiling import st_profile_report

ModuleNotFoundError: No module named 'ydata_profiling'

In [119]:
pip install "pydantic<2.0"

  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.10.6
    Uninstalling pydantic-2.10.6:Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.5.1 requires numpy<1.24,>=1.16.0, but you have numpy 1.24.3 which is incompatible.
pydantic-settings 2.8.0 requires pydantic>=2.7.0, but you have pydantic 1.10.21 which is incompatible.
ollama 0.4.7 requires pydantic<3.0.0,>=2.9.0, but you have pydantic 1.10.21 which is incompatible.



      Successfully uninstalled pydantic-2.10.6


In [121]:
from pydantic_settings import BaseSettings

In [123]:
pip uninstall -y pydantic ydata-profiling pandas-profiling dacite

Found existing installation: pydantic 1.10.21
Uninstalling pydantic-1.10.21:
  Successfully uninstalled pydantic-1.10.21
Found existing installation: ydata-profiling 4.5.1
Uninstalling ydata-profiling-4.5.1:
  Successfully uninstalled ydata-profiling-4.5.1
Found existing installation: pandas-profiling 3.6.6
Uninstalling pandas-profiling-3.6.6:
  Successfully uninstalled pandas-profiling-3.6.6
Found existing installation: dacite 1.8.1
Uninstalling dacite-1.8.1:
  Successfully uninstalled dacite-1.8.1
Note: you may need to restart the kernel to use updated packages.


In [124]:
pip install "pydantic<2.0" dacite==1.6.0 ydata-profiling==4.0.0 streamlit-pandas-profiling

  Using cached pydantic-1.10.21-cp38-cp38-win_amd64.whl (2.3 MB)



ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\Akshat Mittu\\AppData\\Roaming\\Python\\Python38\\site-packages\\~umpy\\.libs\\libopenblas64__v0.3.21-gcc_10_3_0.dll'
Consider using the `--user` option or check the permissions.



  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.3
    Uninstalling numpy-1.24.3:
      Successfully uninstalled numpy-1.24.3


In [127]:
pip install ydata-profiling==4.0.0 --user

  Using cached ydata_profiling-4.0.0-py2.py3-none-any.whl (344 kB)


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\Akshat Mittu\\AppData\\Roaming\\Python\\Python38\\site-packages\\~atplotlib.libs\\msvcp140-456d948669199b545d061b84c160bebc.dll'
Check the permissions.




  Using cached pydantic-1.10.21-cp38-cp38-win_amd64.whl (2.3 MB)
  Using cached pandas-1.5.3-cp38-cp38-win_amd64.whl (11.0 MB)
  Using cached statsmodels-0.13.5-cp38-cp38-win_amd64.whl (9.2 MB)
  Using cached matplotlib-3.6.3-cp38-cp38-win_amd64.whl (7.2 MB)
  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3.7.5
    Uninstalling matplotlib-3.7.5:
      Successfully uninstalled matplotlib-3.7.5


In [128]:
from ydata_profiling import ProfileReport
from streamlit_pandas_profiling import st_profile_report

ModuleNotFoundError: No module named 'ydata_profiling'

In [131]:
pip uninstall pydantic

^C
Note: you may need to restart the kernel to use updated packages.


In [132]:
pip install "pydantic==1.10.12"

  Attempting uninstall: pydantic
    Found existing installation: pydantic 1.10.21
    Uninstalling pydantic-1.10.21:
      Successfully uninstalled pydantic-1.10.21
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'D:\\Anaconda\\Lib\\site-packages\\~ydantic\\annotated_types.cp38-win_amd64.pyd'
Consider using the `--user` option or check the permissions.

